In [1]:
import pandas as pd
df = pd.read_csv("data/raw/olist_orders_dataset.csv")
print(df.shape)   # (99441, 8)이면 성공

(99441, 8)


In [2]:
import duckdb
import pandas as pd

con = duckdb.connect("data/interim/olist.duckdb")   # 파일로 저장됨
RAW = "data/raw/"

In [3]:
def load(name, fname, types=None):
    opt = f", types={types}" if types else ""
    con.execute(f"CREATE OR REPLACE VIEW {name} AS "
                f"SELECT * FROM read_csv('{RAW}{fname}', header=true{opt})")

load("orders",         "olist_orders_dataset.csv")
load("order_items",    "olist_order_items_dataset.csv")
load("order_reviews",  "olist_order_reviews_dataset.csv")
load("order_payments", "olist_order_payments_dataset.csv")
load("products",       "olist_products_dataset.csv")
load("cat_tr",         "product_category_name_translation.csv")
# 우편번호는 파일에 따옴표가 붙어 있어 문자열로 읽힘 → 정수로 지정 (조인 키 타입 통일)
load("customers",   "olist_customers_dataset.csv",   {"customer_zip_code_prefix": "INTEGER"})
load("sellers",     "olist_sellers_dataset.csv",     {"seller_zip_code_prefix": "INTEGER"})
load("geolocation", "olist_geolocation_dataset.csv", {"geolocation_zip_code_prefix": "INTEGER"})

In [4]:
print(con.execute("SHOW TABLES").df().name.tolist())

for t, c in [("customers","customer_zip_code_prefix"),
             ("sellers","seller_zip_code_prefix"),
             ("geolocation","geolocation_zip_code_prefix")]:
    print(t, con.execute(f"SELECT typeof({c}) FROM {t} LIMIT 1").fetchone()[0])

['cat_tr', 'customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers']
customers INTEGER
sellers INTEGER
geolocation INTEGER


In [5]:
tables = ["orders","order_items","order_reviews","order_payments",
          "customers","sellers","products","geolocation","cat_tr"]
sql = " UNION ALL ".join(f"SELECT '{t}' AS tbl, COUNT(*) AS n_rows FROM {t}" for t in tables)
con.execute(sql).df()

,tbl,n_rows
0,orders,99441
1,order_items,112650
2,order_reviews,99224
3,order_payments,103886
4,customers,99441
5,sellers,3095
6,products,32951
7,geolocation,1000163
8,cat_tr,71


In [6]:
con.execute("DESCRIBE orders").df()[["column_name", "column_type"]]

,column_name,column_type
0,order_id,VARCHAR
1,customer_id,VARCHAR
2,order_status,VARCHAR
3,order_purchase_timestamp,TIMESTAMP
4,order_approved_at,TIMESTAMP
5,order_delivered_carrier_date,TIMESTAMP
6,order_delivered_customer_date,TIMESTAMP
7,order_estimated_delivery_date,TIMESTAMP


In [7]:
con.execute("""
SELECT COUNT(*)                                   AS n_rows,
       COUNT(*) - COUNT(review_comment_title)     AS n_title_null,
       COUNT(*) - COUNT(review_comment_message)   AS n_msg_null,
       ROUND(100.0 * (COUNT(*) - COUNT(review_comment_message)) / COUNT(*), 1) AS pct_msg_null
FROM order_reviews
""").df()

,n_rows,n_title_null,n_msg_null,pct_msg_null
0,99224,87656,58247,58.7


In [9]:
def null_profile(table):
    cols = con.execute(f"SELECT column_name FROM (DESCRIBE SELECT * FROM {table})").df().column_name
    sql = " UNION ALL ".join(
        f'SELECT \'{c}\' AS col, COUNT(*) - COUNT("{c}") AS n_null, COUNT(*) AS n_rows FROM {table}'
        for c in cols)
    df = con.execute(sql).df()
    df["pct_null"] = (df.n_null / df.n_rows * 100).round(1)
    return df[df.n_null > 0][["col", "n_null", "pct_null"]]

for t in ["orders","order_items","order_reviews","order_payments",
          "customers","sellers","products","geolocation","cat_tr"]:
    res = null_profile(t)
    print(f"\n[{t}]")
    print(res.to_string(index=False) if len(res) else "결측 없음")


[orders]
                          col  n_null  pct_null
            order_approved_at     160       0.2
 order_delivered_carrier_date    1783       1.8
order_delivered_customer_date    2965       3.0

[order_items]
결측 없음

[order_reviews]
                   col  n_null  pct_null
  review_comment_title   87656      88.3
review_comment_message   58247      58.7

[order_payments]
결측 없음

[customers]
결측 없음

[sellers]
결측 없음

[products]
                       col  n_null  pct_null
     product_category_name     610       1.9
       product_name_lenght     610       1.9
product_description_lenght     610       1.9
        product_photos_qty     610       1.9
          product_weight_g       2       0.0
         product_length_cm       2       0.0
         product_height_cm       2       0.0
          product_width_cm       2       0.0

[geolocation]
결측 없음

[cat_tr]
결측 없음


In [10]:
con.execute("""
SELECT order_status,
       COUNT(*)                                        AS n_orders,
       COUNT(order_delivered_customer_date)            AS n_has_date,
       COUNT(*) - COUNT(order_delivered_customer_date) AS n_null_date
FROM orders
GROUP BY order_status
ORDER BY n_orders DESC
""").df()

,order_status,n_orders,n_has_date,n_null_date
0,delivered,96478,96470,8
1,shipped,1107,0,1107
2,canceled,625,6,619
3,unavailable,609,0,609
4,invoiced,314,0,314
5,processing,301,0,301
6,created,5,0,5
7,approved,2,0,2


In [11]:
con.execute("""
CREATE OR REPLACE TABLE rev AS
SELECT * EXCLUDE rn FROM (
  SELECT *, ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY review_answer_timestamp DESC, review_creation_date DESC, review_id) AS rn
  FROM order_reviews)
WHERE rn = 1
""")
con.execute("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_orders FROM rev").df()

,n_rows,n_orders
0,98673,98673


In [12]:
con.execute("""
SELECT COUNT(*)                                        AS n_window,
       COUNT(order_delivered_customer_date)            AS n_delivered,
       ROUND(100.0 * SUM(CASE WHEN CAST(order_delivered_customer_date AS DATE)
                                  > CAST(order_estimated_delivery_date AS DATE)
                              THEN 1 ELSE 0 END)
             / COUNT(order_delivered_customer_date), 2) AS late_pct
FROM orders
WHERE order_purchase_timestamp >= '2017-01-01'
  AND order_purchase_timestamp <  '2018-09-01'
""").df()

,n_window,n_delivered,late_pct
0,99092,96204,6.79


In [13]:
con.execute("""
CREATE OR REPLACE TABLE chk AS
SELECT o.order_id,
       CASE WHEN CAST(o.order_delivered_customer_date AS DATE)
                 > CAST(o.order_estimated_delivery_date AS DATE) THEN 1 ELSE 0 END AS late,
       r.review_score,
       CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END AS low,
       r.review_comment_message
FROM orders o JOIN rev r USING (order_id)
WHERE o.order_delivered_customer_date IS NOT NULL
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp <  '2018-09-01'
""")

# (1) 전체 비율
display(con.execute("""
SELECT COUNT(*) AS n_orders,
       ROUND(100.0 * AVG(low), 1) AS low_pct,
       ROUND(100.0 * COUNT(review_comment_message) / COUNT(*), 1) AS comment_pct
FROM chk""").df())

# (2) 지연 × 저평점
display(con.execute("""
SELECT late, COUNT(*) AS n, SUM(low) AS n_low, ROUND(100.0 * AVG(low), 1) AS low_pct
FROM chk GROUP BY late ORDER BY late""").df())

# (3) 3단계 대상: 지연 아님 & 저평점
display(con.execute("""
SELECT COUNT(*) AS n_stage3, COUNT(review_comment_message) AS n_with_comment
FROM chk WHERE late = 0 AND low = 1""").df())

# (4) 멀티셀러 주문
display(con.execute("""
SELECT COUNT(*) AS n_orders_with_items,
       SUM(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END) AS n_multi,
       ROUND(100.0 * AVG(CASE WHEN n_sellers > 1 THEN 1 ELSE 0 END), 2) AS multi_pct
FROM (SELECT order_id, COUNT(DISTINCT seller_id) AS n_sellers
      FROM order_items GROUP BY order_id)""").df())

,n_orders,low_pct,comment_pct
0,95561,12.8,40.5


,late,n,n_low,low_pct
0,0,89182,8247.0,9.2
1,1,6379,3981.0,62.4


,n_stage3,n_with_comment
0,8247,6449


,n_orders_with_items,n_multi,multi_pct
0,98666,1278.0,1.3
